In [1]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import scanpy as sc

import networkx as nx
import community as community_louvain
from ensemble_helper import *
import os
import argparse

/nfs/home/students/s.reif/miniforge3/envs/sdd-metrics-real/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/s.reif/miniforge3/envs/sdd-metrics-real/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/s.reif/miniforge3/envs/sdd-metrics-real/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/students/s.reif/miniforge3/envs/sdd-metrics-real/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/nfs/home/

In [3]:
adata = sc.read_h5ad('/nfs/home/students/s.reif/SysMed/SysBioSDD/evaluation_metrics/examples_anndata/graphst_visium_mouse_brain_coronal.h5ad')
adata

AnnData object with n_obs × n_vars = 704 × 16562
    obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_MT', 'log1p_total_counts_MT', 'pct_counts_MT', 'n_counts', 'leiden', 'cluster', 'features_summary_cluster', 'features_histogram_cluster', 'features_texture_cluster', 'mclust', 'domain'
    var: 'gene_ids', 'feature_types', 'genome', 'MT', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'cluster_co_occurrence', 'cluster_colors', 'cluster_ligrec', 'cluster_nhood_enrichment', 'domain_colors', 'hvg', 'leiden', 'leiden_colors', 'moranI', 'neighbors', 'pca', 'spatial', 'spatial_neighbors', 'umap'
    obsm: 

### Generating real good anndata for testing

In [4]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.io import mmread
import anndata as ad

# -------- paths --------
cell_data_path   = "/nfs/data3/mopitas/sddproj/data/dlpfc/DLPFC_151671_cell_data.csv"
gene_data_path   = "/nfs/data3/mopitas/sddproj/data/dlpfc/DLPFC_151671_gene_data.csv"
mtx_path         = "/nfs/data3/mopitas/sddproj/data/dlpfc/DLPFC_151671_rawcounts.mtx"
spatial_path     = "/nfs/data3/mopitas/sddproj/data/dlpfc/DLPFC_151671_spatial_coords.csv"
final_clusters_path = "/nfs/data3/mopitas/sddproj/data/dlpfc/151671_sample_final_clusters.csv"  # your uploaded file

# -------- 1) read cell_data (obs) --------
# First column is barcodes (looks like "AAACAAGTATCTCCCA-1")
obs = pd.read_csv(cell_data_path, index_col=0)
obs.index = obs.index.astype(str)
obs.index.name = "barcode"

# -------- 2) read gene_data (var) --------
# First column is ENSG id (your example shows it as the rowname)
var = pd.read_csv(gene_data_path, index_col=0)
var.index = var.index.astype(str)
var.index.name = "gene"

# Optional: if you prefer gene symbols as var_names:
# var_names = var["gene_name"].astype(str).values
# (only do this if unique; otherwise keep ENSG ids)
# if len(pd.Index(var_names).unique()) == len(var_names):
#     var = var.copy()
#     var["gene_ids"] = var.index
#     var.index = var_names
#     var.index.name = "gene_name"

# -------- 3) read rawcounts.mtx --------
X = mmread(mtx_path).tocsr()

# rawcounts.mtx is often (n_genes x n_cells) -> transpose to (cells x genes)
if X.shape == (var.shape[0], obs.shape[0]):
    X = X.T.tocsr()
elif X.shape == (obs.shape[0], var.shape[0]):
    X = X.tocsr()
else:
    raise ValueError(
        f"MTX shape {X.shape} does not match obs {obs.shape[0]} and var {var.shape[0]}.\n"
        f"Expected either (genes,cells)=({var.shape[0]},{obs.shape[0]}) or (cells,genes)=({obs.shape[0]},{var.shape[0]})."
    )

# -------- 4) build AnnData --------
adata = ad.AnnData(X=X, obs=obs, var=var)
adata.obs_names = adata.obs_names.astype(str)
adata.var_names = adata.var_names.astype(str)

# -------- 5) add spatial coords to obsm['spatial'] --------
spatial = pd.read_csv(spatial_path, index_col=0)
spatial.index = spatial.index.astype(str)

# ensure same order as adata.obs_names
spatial = spatial.reindex(adata.obs_names)

# store as (row, col) like Scanpy expects; your file is (col,row) in names
adata.obsm["spatial"] = spatial[["pxl_row_in_fullres", "pxl_col_in_fullres"]].to_numpy()

# optionally keep the raw columns in obs too (handy)
adata.obs["pxl_row_in_fullres"] = spatial["pxl_row_in_fullres"].values
adata.obs["pxl_col_in_fullres"] = spatial["pxl_col_in_fullres"].values

# -------- 6) attach method cluster labels (final_clusters.csv) safely --------
clusters = pd.read_csv(final_clusters_path)

# Case A: clusters file already has a barcode column -> merge by barcode
barcode_cols = [c for c in clusters.columns if c.lower() in {"barcode", "barcodes", "spot", "spot_id"}]
if barcode_cols:
    bcol = barcode_cols[0]
    clusters[bcol] = clusters[bcol].astype(str)
    clusters = clusters.set_index(bcol)

    # left-join keeps all adata cells, missing labels become NaN
    for col in clusters.columns:
        adata.obs[col] = clusters.reindex(adata.obs_names)[col].values

else:
    # Case B: your example has Cell_ID = "Cell_1", "Cell_2", ... (no barcode)
    # We create the same Cell_ID in obs by row order and merge on that.
    adata.obs["Cell_ID"] = [f"Cell_{i}" for i in range(1, adata.n_obs + 1)]
    clusters["Cell_ID"] = clusters["Cell_ID"].astype(str)
    clusters = clusters.set_index("Cell_ID")

    # join by Cell_ID
    joined = adata.obs[["Cell_ID"]].join(clusters, on="Cell_ID", how="left")

    # attach all cluster columns (except Cell_ID itself)
    for col in clusters.columns:
        adata.obs[col] = joined[col].values

    # Report how many are missing (this is where your 4109 vs 4110 shows up)
    missing = joined[clusters.columns].isna().all(axis=1).sum()
    print(f"[INFO] Cells without cluster labels after join: {missing} / {adata.n_obs}")

    # Optionally: if you want to DROP the unmatched cell(s) instead of keeping NaNs:
    # keep = ~joined[clusters.columns].isna().all(axis=1)
    # adata = adata[keep].copy()

# -------- 7) final sanity prints --------
print(adata)
print("X shape:", adata.shape)
print("obsm['spatial'] shape:", adata.obsm["spatial"].shape)

# -------- 8) save --------
#adata.write_h5ad("DLPFC_151671_from_csv_mtx.h5ad")
#print("Saved:", "DLPFC_151671_from_csv_mtx.h5ad")


[INFO] Cells without cluster labels after join: 1 / 4110
AnnData object with n_obs × n_vars = 4110 × 33538
    obs: 'sample_id', 'Cluster', 'sum_umi', 'sum_gene', 'subject', 'position', 'replicate', 'subject_position', 'discard', 'key', 'cell_count', 'SNN_k50_k4', 'SNN_k50_k5', 'SNN_k50_k6', 'SNN_k50_k7', 'SNN_k50_k8', 'SNN_k50_k9', 'SNN_k50_k10', 'SNN_k50_k11', 'SNN_k50_k12', 'SNN_k50_k13', 'SNN_k50_k14', 'SNN_k50_k15', 'SNN_k50_k16', 'SNN_k50_k17', 'SNN_k50_k18', 'SNN_k50_k19', 'SNN_k50_k20', 'SNN_k50_k21', 'SNN_k50_k22', 'SNN_k50_k23', 'SNN_k50_k24', 'SNN_k50_k25', 'SNN_k50_k26', 'SNN_k50_k27', 'SNN_k50_k28', 'GraphBased', 'Maynard', 'Martinowich', 'layer_guess', 'layer_guess_reordered', 'layer_guess_reordered_short', 'expr_chrM', 'expr_chrM_ratio', 'SpatialDE_PCA', 'SpatialDE_pool_PCA', 'HVG_PCA', 'pseudobulk_PCA', 'markers_PCA', 'SpatialDE_UMAP', 'SpatialDE_pool_UMAP', 'HVG_UMAP', 'pseudobulk_UMAP', 'markers_UMAP', 'SpatialDE_PCA_spatial', 'SpatialDE_pool_PCA_spatial', 'HVG_PCA_sp

In [2]:
adata_hist = sc.read_h5ad('/nfs/home/students/s.reif/SysMed/SysBioSDD/evaluation_metrics/dlpfc/anndata/DLPFC_151671_anndata_with_histology_clusters.h5ad')
adata_hist

AnnData object with n_obs × n_vars = 4110 × 33538
    obs: 'sample_id', 'Cluster', 'sum_umi', 'sum_gene', 'subject', 'position', 'replicate', 'subject_position', 'discard', 'key', 'cell_count', 'SNN_k50_k4', 'SNN_k50_k5', 'SNN_k50_k6', 'SNN_k50_k7', 'SNN_k50_k8', 'SNN_k50_k9', 'SNN_k50_k10', 'SNN_k50_k11', 'SNN_k50_k12', 'SNN_k50_k13', 'SNN_k50_k14', 'SNN_k50_k15', 'SNN_k50_k16', 'SNN_k50_k17', 'SNN_k50_k18', 'SNN_k50_k19', 'SNN_k50_k20', 'SNN_k50_k21', 'SNN_k50_k22', 'SNN_k50_k23', 'SNN_k50_k24', 'SNN_k50_k25', 'SNN_k50_k26', 'SNN_k50_k27', 'SNN_k50_k28', 'GraphBased', 'Maynard', 'Martinowich', 'layer_guess', 'layer_guess_reordered', 'layer_guess_reordered_short', 'expr_chrM', 'expr_chrM_ratio', 'SpatialDE_PCA', 'SpatialDE_pool_PCA', 'HVG_PCA', 'pseudobulk_PCA', 'markers_PCA', 'SpatialDE_UMAP', 'SpatialDE_pool_UMAP', 'HVG_UMAP', 'pseudobulk_UMAP', 'markers_UMAP', 'SpatialDE_PCA_spatial', 'SpatialDE_pool_PCA_spatial', 'HVG_PCA_spatial', 'pseudobulk_PCA_spatial', 'markers_PCA_spatial', 

In [3]:
adata_hist.obs

,sample_id,Cluster,sum_umi,sum_gene,subject,position,replicate,subject_position,discard,key,...,BayesSpace,BASS,GraphST,stLearn,SCAN-IT,bass,scanit,bayesspace,stlearn,graphst
barcode,,,,,,,,,,,,,,,,,,,,,
AAACAAGTATCTCCCA-1,151671,5,9080,3646,Br5595,300,1,Br5595_pos300,False,151671_AAACAAGTATCTCCCA-1,...,5.0,1.0,2.0,1.0,1.0,3,5,4,3,1
AAACACCAATAACTGC-1,151671,5,6183,2734,Br5595,300,1,Br5595_pos300,False,151671_AAACACCAATAACTGC-1,...,3.0,1.0,1.0,1.0,3.0,6,2,2,3,2
AAACAGAGCGACTCCT-1,151671,6,6273,2857,Br5595,300,1,Br5595_pos300,False,151671_AAACAGAGCGACTCCT-1,...,5.0,6.0,2.0,4.0,3.0,4,6,7,2,8
AAACAGCTTTCAGAAG-1,151671,7,7015,2921,Br5595,300,1,Br5595_pos300,False,151671_AAACAGCTTTCAGAAG-1,...,5.0,1.0,2.0,3.0,2.0,3,0,4,0,1
AAACAGGGTCTATATT-1,151671,1,6991,3035,Br5595,300,1,Br5595_pos300,False,151671_AAACAGGGTCTATATT-1,...,2.0,3.0,2.0,4.0,1.0,3,0,4,3,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTGTTTCACATCCAGG-1,151671,4,2882,1681,Br5595,300,1,Br5595_pos300,False,151671_TTGTTTCACATCCAGG-1,...,3.0,5.0,2.0,2.0,2.0,8,2,4,0,2
TTGTTTCATTAGTCTA-1,151671,4,4080,2061,Br5595,300,1,Br5595_pos300,False,151671_TTGTTTCATTAGTCTA-1,...,3.0,6.0,1.0,2.0,2.0,8,2,4,0,2
TTGTTTCCATACAACT-1,151671,2,3063,1596,Br5595,300,1,Br5595_pos300,False,151671_TTGTTTCCATACAACT-1,...,4.0,4.0,2.0,4.0,1.0,3,7,4,4,1


In [4]:
# write anndata
#adata.write_h5ad("/nfs/home/students/s.reif/SysMed/SysBioSDD/evaluation_metrics/examples_dlpfc/DLPFC_151671_anndata.h5ad")
adata_hist.obs['Cluster'].to_csv("/nfs/home/students/s.reif/SysMed/SysBioSDD/evaluation_metrics/dlpfc/csvs/groundtruth.csv")
adata_hist.obs['bayesspace'].to_csv("/nfs/home/students/s.reif/SysMed/SysBioSDD/evaluation_metrics/dlpfc/csvs/bayespace.csv")
adata_hist.obs['bass'].to_csv("/nfs/home/students/s.reif/SysMed/SysBioSDD/evaluation_metrics/dlpfc/csvs/bass.csv")
adata_hist.obs['graphst'].to_csv("/nfs/home/students/s.reif/SysMed/SysBioSDD/evaluation_metrics/dlpfc/csvs/graphst.csv")
adata_hist.obs['stlearn'].to_csv("/nfs/home/students/s.reif/SysMed/SysBioSDD/evaluation_metrics/dlpfc/csvs/stlearn.csv")
adata_hist.obs['scanit'].to_csv("/nfs/home/students/s.reif/SysMed/SysBioSDD/evaluation_metrics/dlpfc/csvs/scanit.csv")